# pRB/RB Ratio Analysis and AnnData Conversion Notebook

This notebook processes single-cell feature data, calculates the **pRB/RB ratio**, and incorporates it into an AnnData object for downstream analyses such as clustering, PHATE, and UMAP. The workflow includes the following steps:

---

### 1. Build Combined DataFrame
- Load per-well single-cell data from CSV files.
- Add metadata (treatment, dose, sample_ID, etc.).
- Concatenate all wells into a single `fullest_df`.

---

### 2. Calculate pRB/RB Ratio
- Compute the nuclear ratio:  
  \[{pRB/RB ratio} = {R0\_pRb\_nuc\_mean}/{R0\_Rb\_nuc\_mean}\]
- Store the ratio in the DataFrame for each cell.

---

### 3. Histogram & Threshold Selection
- Z-normalize the ratio distribution.  
- Plot histogram of ratios to visually inspect population structure.  
- Choose either:
  - **Single cutoff** → split into `pRB_low` and `pRB_high`.  
  - **Double cutoff** → split into `pRB_low`, `pRB_middle`, and `pRB_high`.  

---

### 4. Save Labels into DataFrame
- Add categorical labels (`pRB_low`, `pRB_high`, etc.) to the metadata.  
- Preserve both the raw ratio and its z-normalized version.

---

### 5. Create AnnData Object
- Separate **metadata** (goes to `.obs`) and **features** (goes to `.X`).  
- Z-normalize all features (including the ratio).  
- Save as `standard_adata.h5ad`.

---

### 6. Subsample with Sketch (Optional)
- Use `sketch` to generate a representative subsample of the data.  
- Save as `standard_adata_sub.h5ad` for faster exploratory analysis.

---

✅ Final products:  
- `standard_adata.h5ad` → full dataset with ratio + labels  
- `standard_adata_sub.h5ad` → subsampled dataset for visualization and quick analysis


In [ ]:
import pickle
import os
import numpy as np
print("PYTHONPATH:", os.environ.get('PYTHONPATH'))
print("PATH:", os.environ.get('PATH'))
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn import preprocessing
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandas as pd
import phate
import math
import random
import gc
import scprep
from datetime import datetime, time
from matplotlib.animation import ImageMagickWriter
import matplotlib.animation as animation
import zipfile
from urllib.request import urlopen
import scipy.stats as st
from scipy.stats import norm
from scipy.stats import gaussian_kde
from scipy.stats import kde
from scipy.stats import binned_statistic
from scipy.stats import f_oneway
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['pdf.fonttype'] = 42
print(sns.__version__)
from anndata import AnnData
import scanpy as sc
from delve import *
import anndata as ad
from sklearn.preprocessing import MinMaxScaler
from sketchKH import sketch
from sklearn.cluster import KMeans
import umap
print(sc.__version__)
today = datetime.now().strftime("%m%d%Y-%H%M")

In [ ]:
import pickle
import os
import numpy as np
print("PYTHONPATH:", os.environ.get('PYTHONPATH'))
print("PATH:", os.environ.get('PATH'))
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn import preprocessing
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandas as pd
print("seaborn and pandas")
import phate
print("phate")
import math
import random
import gc
import scprep
from datetime import datetime, time
from matplotlib.animation import ImageMagickWriter
import matplotlib.animation as animation
import zipfile
print("matplotlib")
from urllib.request import urlopen
import scipy.stats as st
from scipy.stats import norm
from scipy.stats import gaussian_kde
from scipy.stats import kde
from scipy.stats import binned_statistic
from scipy.stats import f_oneway
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['pdf.fonttype'] = 42
print(sns.__version__)
from anndata import AnnData
print("scipy")
import scanpy as sc
print("scanpy")
from delve import *
import anndata as ad
print("anndata")
from sklearn.preprocessing import MinMaxScaler
from sketchKH import sketch
from sklearn.cluster import KMeans
import umap
print(sc.__version__)
today = datetime.now().strftime("%m%d%Y-%H%M")

In [ ]:
### full_dir should be the directory that contains the output of your "calculate cell properties" notebook. Called cell_data by default
### well_list should be every well in the dataset that you intend to combine into a single adata object for analysis

full_dir = r'Z:\Omar\4i_modernize_test_B02\cell_data'
well_list = ['B02']



In [ ]:
# Definition to Normalize the dataframe by z-score

def standardizeColumns(df):
    return (df - df.mean()) / df.std()

I Like to name my plate here so i know what doses and treatments these are for a specific plate/experiment

In [ ]:
#Builds a dataframe for each well defined in well_list, then outputs the length of that list. The number output is how many cells were found in the well

fullest_df = pd.DataFrame()
for well in well_list:
    print(f'starting Well {well}')
    full_df = pd.read_csv(os.path.join(full_dir,f'cell_data_{well}_df.csv'), sep=',') 

   ### This part here is optional, you can add important metadata information at this step. Treatment and Sample ID are shown as examples

    if 'B' in well and ('02' in well):
        full_df['treatment'] = 'control'        # Type in dose of drug/treatment
        full_df['treatment time'] = '96h'    # name of drug/treatment ex: Control, Abema, Palbo, Nutlin...etc
        full_df['sample_ID'] = 1       # Sample ID will group together the wells with the same treatment conditions like duplicate and triplicate conditions on your plate so if your treatment/conditions in B02 are the same as C02 or another well, give them the same sample ID and group them to be the same so the computer isnt working with 60 wells or 60 separate things, its rather using one sample ID to make for example 20 things to work with instead.
       # full_df['Group'] = 1
    if 'B' in well and ('03' in well):
        full_df['treatment'] = 'abema_fulv'
        full_df['treatment time'] = '96h'
        full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    if 'C' in well and ('04' in well):
         full_df['treatment'] = 'abema_fulv'
         full_df['treatment time'] = '72h'
         full_df['sample_ID'] = 3
        # full_df['Group'] = 1
    if 'B' in well and ('05' in well):
         full_df['treatment'] = 'abema_fulv'
         full_df['treatment time'] = '48h'
         full_df['sample_ID'] = 4
         #full_df['Group'] = 1
    if 'B' in well and ('06' in well):
         full_df['treatment'] = 'abema_fulv'
         full_df['treatment time'] = '24h'
         full_df['sample_ID'] = 5
         #full_df['Group'] = 1

    #if 'C' in well and ('02' in well):
        #full_df['dilution'] = '1:1000'
        #full_df['treatment'] = 'control' 
        # full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    #if 'C' in well and ('03' in well):
        #full_df['dilution'] = '1:1000'
        #full_df['treatment'] = '150nM palbo'
        # full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    # if 'C' in well and ('05' in well):
    #     full_df['dose'] = '100nM + 100nM'
    #     full_df['treatment'] = 'Fulv + Palbo'
    #     full_df['sample_ID'] = 3
    #    # full_df['Group'] = 1
    # if 'C' in well and ('07' in well):
    #     full_df['dose'] = '100nM + 200nM'
    #     full_df['treatment'] = 'Fulv + Palbo'
    #     full_df['sample_ID'] = 4
    #     #full_df['Group'] = 1
    # if 'C' in well and ('09' in well):
    #     full_df['dose'] = '100nM + 500nM'
    #     full_df['treatment'] = 'Fulv + Palbo'
    #     full_df['sample_ID'] = 5
    #     #full_df['Group'] = 1
    
    #if 'D' in well and ('02' in well):
        #full_df['dilution'] = '1:800'
        #full_df['treatment'] = 'control' 
        # full_df['sample_ID'] = 1
       # full_df['Group'] = 1
    #if 'D' in well and ('03' in well):
        #full_df['dilution'] = '1:800'
        #full_df['treatment'] = '150nM palbo'
        # full_df['sample_ID'] = 2
       # full_df['Group'] = 1
    # if 'D' in well and ('05' in well):
    #     full_df['dose'] = '100nM + 100nM'
    #     full_df['treatment'] = 'Fulv + Palbo'
    #     full_df['sample_ID'] = 3
    #    # full_df['Group'] = 1
    # if 'D' in well and ('07' in well):
    #     full_df['dose'] = '100nM + 200nM'
    #     full_df['treatment'] = 'Fulv + Palbo'
    #     full_df['sample_ID'] = 4
    #     #full_df['Group'] = 1
    # if 'D' in well and ('09' in well):
    #     full_df['dose'] = '100nM + 500nM'
    #     full_df['treatment'] = 'Fulv + Palbo'
    #     full_df['sample_ID'] = 5
    #     #full_df['Group'] = 1
    
    #if 'E' in well and ('02' in well):
        #full_df['dilution'] = '1:500'
        #full_df['treatment'] = 'control' 
        # full_df['sample_ID'] = 6
       # full_df['Group'] = 1
   # if 'E' in well and ('03' in well):
        #full_df['dilution'] = '1:500'
        #full_df['treatment'] = '150nM palbo'
        # full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    # if 'E' in well and ('05' in well):
    #     full_df['dose'] = '10nM + 100nM + 100nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 8
    #    # full_df['Group'] = 1
    # if 'E' in well and ('07' in well):
    #     full_df['dose'] = '10nM + 100nM + 200nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 9
    #     #full_df['Group'] = 1
    # if 'E' in well and ('09' in well):
    #     full_df['dose'] = '10nM + 100nM + 500nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 10
    #     #full_df['Group'] = 1

    #if 'F' in well and ('02' in well):
        #full_df['dilution'] = '1:300'
        #full_df['treatment'] = 'control' 
        # full_df['sample_ID'] = 6
       # full_df['Group'] = 1
    #if 'F' in well and ('03' in well):
        #full_df['dilution'] = '1:300'
        #full_df['treatment'] = '150nM palbo'
        # full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    # if 'F' in well and ('05' in well):
    #     full_df['dose'] = '10nM + 100nM + 100nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 8
    #    # full_df['Group'] = 1
    # if 'F' in well and ('07' in well):
    #     full_df['dose'] = '10nM + 100nM + 200nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 9
    #     #full_df['Group'] = 1
    # if 'F' in well and ('09' in well):
    #     full_df['dose'] = '10nM + 100nM + 500nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 10
    #     #full_df['Group'] = 1
    
    #if 'G' in well and ('02' in well):
        #full_df['dilution'] = '1:100'
        #full_df['treatment'] = 'control' 
        # full_df['sample_ID'] = 6
       # full_df['Group'] = 1
    #if 'G' in well and ('03' in well):
        #full_df['dilution'] = '1:100'
        #full_df['treatment'] = '150nM palbo'
        # full_df['sample_ID'] = 7
       # full_df['Group'] = 1
    # if 'G' in well and ('05' in well):
    #     full_df['dose'] = '10nM + 100nM + 100nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 8
    #    # full_df['Group'] = 1
    # if 'G' in well and ('07' in well):
    #     full_df['dose'] = '10nM + 100nM + 200nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 9
    #     #full_df['Group'] = 1
    # if 'G' in well and ('09' in well):
    #     full_df['dose'] = '10nM + 100nM + 500nM'
    #     full_df['treatment'] = 'Gemc + Fulv + Palbo'
    #     full_df['sample_ID'] = 10
    #     #full_df['Group'] = 1

    exec(f'well{well}_df = full_df.copy()')
    fullest_df = pd.concat([fullest_df, full_df], ignore_index = True)
    print(len(full_df))
    print(len(fullest_df))

### You may find that you need to force some of the columns added this way into specific data types. You can do that here.
# Convert sample_ID column to categorical data type
fullest_df['sample_ID'] = fullest_df['sample_ID'].astype('category')
fullest_df['treatment'] = fullest_df['treatment'].astype(str)
fullest_df['treatment time'] = fullest_df['treatment time'].astype(str)

In [ ]:
fullest_df.columns

In [ ]:
### It is good practice to save these dataframes at key steps, such as this one
fullest_df.to_csv(r'Z:\Omar\4i_modernize_test_B02\fullest_df.csv')


In [ ]:
import pandas as pd

# Identify columns that contain any NaN values
columns_with_nan = fullest_df.columns[fullest_df.isnull().any()].tolist()

# Print the list of column headers with NaN values
print("Columns with NaN values:", columns_with_nan)

In [ ]:

# Definition to Normalize the dataframe by z-score

def standardizeColumns(df):
    return (df - df.mean()) / df.std()

Adding in calculated features into fullest_df

In [ ]:
fullest_df["pRB_RB_ratio"] = fullest_df["R0_pRB_nuc_mean"] / fullest_df["R0_RB_nuc_mean"]

In [ ]:
# not in this experiment's antibody panel: fullest_df["total_pS6"] = (fullest_df["R7_pS6_nuc_mean"] + fullest_df["R7_pS6_ring_mean"])

In [ ]:
# not in this experiment's antibody panel: fullest_df["total_S6"] = (fullest_df["R7_S6_nuc_mean"] + fullest_df["R7_S6_ring_mean"])

In [ ]:
# not in this experiment's antibody panel: fullest_df["total_pS6_S6_ratio"] = fullest_df["total_pS6"] / fullest_df["total_S6"]

In [ ]:
fullest_df["total_area_DNA"] = fullest_df["R0_DNA_nuc_mean"] * fullest_df["nuc_area"]

In [ ]:
# not in this experiment's antibody panel: fullest_df["pAKT_AKT_ratio"] = fullest_df["R5_pAKT_nuc_mean"] / fullest_df["R5_AKT_nuc_mean"]

In [ ]:
# not in this experiment's antibody panel: fullest_df["pCDK2_CDK2_ratio"] = fullest_df["R1_CDK2pT160_nuc_mean"] / fullest_df["R1_CDK2_nuc_mean"]

Generate histogram to determins pRB/RB ratio threshold

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

FEATURE = "pRB_RB_ratio"

# Extract only sample 4
vals = np.asarray(
    fullest_df.loc[fullest_df["sample_ID"] == 1, FEATURE].values
).ravel()

# Fit scaler on the FULL dataset
all_vals = np.asarray(fullest_df[FEATURE].values).ravel()
_scaler_full = StandardScaler().fit(all_vals.reshape(-1, 1))

# Z-score only sample 4 using the full-dataset scaler
vals_z = _scaler_full.transform(vals.reshape(-1, 1)).ravel()

print("Sample 5 z-scoring stats:")
print("  N    =", vals.size)
print("  mean =", float(_scaler_full.mean_[0]))
print("  std  =", float(_scaler_full.scale_[0]))


In [ ]:
# You can adjust the number of bins here
N_BINS    = 70
FIGSIZE   = (16, 7)
BAR_ALPHA = 0.85
EDGE_LW   = 0.8

# Build histogram on Z-SCORED data (so cutoffs are in z-space)
counts, edges = np.histogram(vals_z, bins=N_BINS)
centers = 0.5 * (edges[:-1] + edges[1:])
widths  = edges[1:] - edges[:-1]


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(centers, counts, width=widths, align="center",
       alpha=BAR_ALPHA, edgecolor="black", linewidth=EDGE_LW)

ymax = max(counts) * 1.15 if counts.size else 1.0
ax.set_ylim(0, ymax)

# Label counts above bars
for cx, cy in zip(centers, counts):
    if cy > 0:
        ax.text(cx, cy + (0.02 * ymax), f"{cy:,}",
                ha="center", va="bottom", fontsize=7, fontweight="bold", rotation=90)

# --- Tick marks at every bin edge, keep 0 as a visible tick ---
all_edges = edges
ax.set_xticks(all_edges)

# Format labels: show fewer decimals, emphasize 0
labels = []
for x in all_edges:
    labels.append("0" if np.isclose(x, 0, atol=1e-6) else f"{x:.2f}")
ax.set_xticklabels(labels, rotation=65, ha="right", fontsize=8)

# <<< Set your cutoff here in Z-SPACE after inspecting the histogram >>>
THRESHOLD_Z = 0.80   # e.g., -0.20 or 0.15, etc.

# Draw the vertical line at your z-cutoff
ax.axvline(THRESHOLD_Z, color="red", linestyle="--", linewidth=1.5)

ax.set_title(f"Histogram of {FEATURE} | N={vals.size} | {N_BINS} bins", fontsize=14)
ax.set_xlabel(f"{FEATURE} (z-scored in standard_adata.h5ad)", fontsize=12)
ax.set_ylabel("Cell count", fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import anndata as ad
from pathlib import Path


ONE threshold cut off point to split cells into high or low pRB/RB

In [ ]:
# --- ONE threshold, label pRB_low / pRB_high on fullest_df ---

FEATURE = "pRB_RB_ratio"

# You chose THRESHOLD_Z on your Z histogram above. Convert Z -> RAW once:
THRESHOLD_Z = 0.80 # <- whatever you picked on the z-hist plot
THRESHOLD_RAW = float(_scaler_full.inverse_transform([[THRESHOLD_Z]]).ravel()[0])

# Label on RAW values (from fullest_df)
vals_raw = fullest_df[FEATURE].astype(float).to_numpy()
labels   = np.where(vals_raw < THRESHOLD_RAW, "pRB_low", "pRB_high")

fullest_df["pRB_label"] = pd.Categorical(labels, categories=["pRB_low","pRB_high"])
print(fullest_df["pRB_label"].value_counts())
print("Cutoff used (RAW):", THRESHOLD_RAW, " | RAW min/max:", float(vals_raw.min()), float(vals_raw.max()))


TWO threshold cut off points for splitting cells into high, medium, and low pRB/RB

In [ ]:
# --- TWO thresholds from Z histogram; map to RAW and label on RAW ---

FEATURE = "pRB_rB_ratio"
THRESHOLD_LOW_Z  = -0.30   # example
THRESHOLD_HIGH_Z =  0.10   # example

THRESHOLD_LOW  = float(_scaler_full.inverse_transform([[THRESHOLD_LOW_Z]]).ravel()[0])
THRESHOLD_HIGH = float(_scaler_full.inverse_transform([[THRESHOLD_HIGH_Z]]).ravel()[0])
if THRESHOLD_LOW > THRESHOLD_HIGH:
    THRESHOLD_LOW, THRESHOLD_HIGH = THRESHOLD_HIGH, THRESHOLD_LOW

vals_raw = fullest_df[FEATURE].astype(float).to_numpy()
labels = np.full(vals_raw.shape[0], "pRB_middle", dtype=object)
labels[vals_raw < THRESHOLD_LOW]  = "pRB_low"
labels[vals_raw > THRESHOLD_HIGH] = "pRB_high"

fullest_df["pRB_label"] = pd.Categorical(labels, categories=["pRB_low","pRB_middle","pRB_high"])
print(fullest_df["pRB_label"].value_counts())
print("RAW cutoffs:", THRESHOLD_LOW, THRESHOLD_HIGH, " | RAW min/max:", float(vals_raw.min()), float(vals_raw.max()))

Saving pRB_high pRB_low column and labels to fullest_df

In [ ]:
# --- Saving pRB_high / pRB_low / pRB_middle labels into fullest_df ---
# note: "labels" must be defined by whichever threshold block you run above

# categorical label column
fullest_df["pRB_label"] = pd.Categorical(labels, categories=np.unique(labels))

# convenience booleans (only if you want them)
if "pRB_low" in labels:
    fullest_df["pRB_low"] = (fullest_df["pRB_label"] == "pRB_low")
if "pRB_middle" in labels:
    fullest_df["pRB_middle"] = (fullest_df["pRB_label"] == "pRB_middle")
if "pRB_high" in labels:
    fullest_df["pRB_high"] = (fullest_df["pRB_label"] == "pRB_high")

# quick counts
print(fullest_df["pRB_label"].value_counts())

Cleaning up fullest_df for conversion to AnnData object

In [ ]:
# --- keep your existing drops as-is ---
fullest_df = fullest_df.drop(columns=['Unnamed: 0', 'bbox-0', 'bbox-1', 'bbox-2', 'bbox-3', 'nuc_mask', 'ring_mask'])
columns_to_drop = ['Unnamed: 0', 'bbox-0', 'bbox-1', 'bbox-2', 'bbox-3', 'nuc_mask', 'ring_mask']

# --- METADATA (goes to .obs): include label, NOT the ratio ---
metadata_cols = ['label', 'well', 'sample_ID', 'treatment', 'treatment time', 'centroid-0', 'centroid-1', 'pRB_label']
metadata = fullest_df[metadata_cols].copy()

# --- FEATURES (go to .X): drop only metadata; KEEP pRB_rB_ratio so it is z-normalized ---
feature_df = fullest_df.drop(columns=metadata_cols, errors='ignore')

In [ ]:
# --- FEATURES (go to .X): drop only metadata; KEEP pRB_rB_ratio so it is z-normalized ---
feature_df = fullest_df.drop(columns=metadata_cols, errors='ignore')

# Keep numeric columns only (avoid strings like 'B02', '10 nM', etc.)
feature_df = feature_df.select_dtypes(include=['number']).copy()

# Z-normalize the features (this now includes pRB_rB_ratio)
standard_df = standardizeColumns(feature_df)

# (Optional) if any columns still had non-numeric noise you coerced earlier:
# feature_df = feature_df.apply(pd.to_numeric, errors='coerce')
# standard_df = standardizeColumns(feature_df).dropna(axis=1, how='any')


In [ ]:
import pandas as pd

# Remove columns that contain any NaN values
print("Shape before removing columns with NaN:", standard_df.shape)
standard_df = standard_df.dropna(axis=1, how='any')

# Print the shape of the DataFrame before and after removing columns with NaN
print("Shape after removing columns with NaN:", standard_df.shape)


Create standard_adata.h5ad

In [ ]:
# Convert the pandas dataframe to an anndata object
standard_adata = ad.AnnData(standard_df)

# Add metadata back to the anndata object
standard_adata.obs = metadata.copy()
standard_adata.obs_names = [f'C_{i}' for i in standard_adata.obs_names]

# NEW: also expose the normalized ratio in .obs (pulled from the z-scored features)
standard_adata.obs["pRB_RB_ratio"] = standard_df["pRB_RB_ratio"].to_numpy()


#Save the entire adata file
adata_save_path = r'Z:\Omar\4i_modernize_test_B02\standard_adata.h5ad'
standard_adata.write_h5ad(adata_save_path)

In [ ]:
#use this block if you want to load in a standard_adata.h5ad file to subsample wihtout creating a new one or overriding one.
import anndata as ad

# Load the existing AnnData file (do NOT recreate or overwrite)
adata_path = r"Z:\Omar\4i_modernize_test_B02\standard_adata.h5ad"
standard_adata = ad.read_h5ad(adata_path)

print(f"Loaded standard AnnData: {standard_adata.shape}")

In [ ]:
###Sketching lets your subsample your data accurately. 
### This example groups for subsampling based on the sample_id metadata

idx, standard_adata_sub = sketch(standard_adata, num_subsamples = 2000, frequency_seed = 42, sample_set_key = 'sample_ID')
#Save the entire adata file
adata_save_path = r'Z:\Omar\4i_modernize_test_B02\standard_adata_sub.h5ad'
standard_adata_sub.write_h5ad(adata_save_path)
print(f'Saved subsampled adata to: {adata_save_path}')

In [ ]:
# Pull metadata
sample_ids = adata.obs['sample_ID']
treatment = adata.obs['treatment']
treatment_time = adata.obs['treatment time']
threshold_col = 'pRB_label'

# Keep original order
selected_sample_ids = [5, 4, 3, 2, 1]    ### change this to match desired order of sample_IDs ###

# Color map
color_map = {
    'pRB_low': 'skyblue',
    'pRB_high': 'salmon'
}

# Iterate through every protein to plot
for protein_name in adata.var_names:
    protein_index = np.where(adata.var_names == protein_name)[0][0]

    data = []
    group_labels = []
    violin_classes = []
    group_times = []  

    for sample_id in selected_sample_ids:
        sample_mask = (sample_ids == sample_id)
        if sample_mask.sum() == 0:
            continue

        this_time_raw = treatment_time[sample_mask].iloc[0]
        # Convert to integer hours (handles "48h" or 48)
        this_time = int(str(this_time_raw).replace('h', ''))
        this_treatment = treatment[sample_mask].iloc[0]

        group_times.append(this_time)  # <-- NEW

        for class_label in ['pRB_low', 'pRB_high']:
            class_mask = (adata.obs[threshold_col] == class_label)
            combined_mask = sample_mask & class_mask

            if combined_mask.sum() == 0:
                data.append([np.nan])
            else:
                values = adata.X[combined_mask, protein_index]
                if hasattr(values, "toarray"):
                    values = values.toarray().flatten()
                else:
                    values = np.asarray(values).flatten()
                data.append(values)

            group_labels.append(
                f"{this_treatment}_{class_label}\nTreatment Time: {this_time}"
            )
            violin_classes.append(class_label)

    # Plot if data exists
    valid_data = [d for d in data if len(d) > 0 and not np.all(np.isnan(d))]
    if valid_data:
        fig, ax1 = plt.subplots(figsize=(16, 8))

        vp = ax1.violinplot(
            data,
            showmeans=False,
            showmedians=True,
            showextrema=False
        )

        # --- NEW: Background shading for 48h and 96h ---
        for i, t in enumerate(group_times):
            if t in [48, 96]:
                start = 2 * i + 0.5
                end = 2 * i + 2.5
                ax1.axvspan(start, end, color='lightgrey', alpha=0.6, zorder=0)

        # Apply colors
        for body, class_label in zip(vp['bodies'], violin_classes):
            body.set_facecolor(color_map[class_label])
            body.set_edgecolor('black')
            body.set_alpha(0.7)
            body.set_zorder(2)  # keep violins above shading

        # Median styling
        if 'cmedians' in vp:
            vp['cmedians'].set_color('black')
            vp['cmedians'].set_linewidth(1.2)
            vp['cmedians'].set_zorder(3)

        ax1.set_xlabel('Drug Treatment and Treatment Time split by pRB label')
        ax1.set_ylabel('Z Normalized Intensity')
        ax1.set_xticks(np.arange(1, len(group_labels) + 1))
        ax1.set_xticklabels(group_labels, rotation=45, ha='right')

        combined_data = np.concatenate(valid_data)
        ax1.set_ylim(np.percentile(combined_data, 1), np.percentile(combined_data, 99))

        # Legend
        legend_elements = [
            Patch(facecolor=color_map['pRB_low'], edgecolor='black', label='pRB_low'),
            Patch(facecolor=color_map['pRB_high'], edgecolor='black', label='pRB_high')
        ]
        ax1.legend(handles=legend_elements, title='pRB label')

        plt.title(f"{protein_name} expression split by pRB_low / pRB_high")
        plt.tight_layout()
        plt.show()

Subsequent notebooks will load in the saved adata file of your choice, either the full dataset of the subsampled dataset.
Additionally, you can produce any desired normalized or non-normalized adata file here. Z-score normalized is the provided example, but other methods can be used with minor modifications. i.e. min-max normalization using the provided code.

In [ ]:
fullest_df.columns

In [ ]:
standard_adata_sub.obs